In [1]:
import torch

### Checking that cuda exists

In [2]:
torch.accelerator.current_accelerator()

device(type='cuda')

In [3]:
torch.__version__

'2.12.1+cu130'

### Tensor gotchas

In [4]:
import numpy as np
import random

# creating a random array of C * W * H
C = 3
H = 128
W = 128 

# random python
data = [[[ random.random() for _ in range(H)] for _ in range(W)] for _ in range(C)]
print(data)

# random numpy
# way 1
np_data = np.array(data, dtype=np.float32) 
print(np_data, np_data.shape)

# way 2
np_data2 = np.random.rand(C, W, H) # np.randint(0, 10, (C, W, H))
print(np_data2, np_data2.shape)

# random tensor
# way 1
tensor_data1 = torch.as_tensor(np_data)
print(tensor_data1, tensor_data1.shape)

# way 2
tensor_data2 = torch.rand((C, W, H)) # similar to numpy
print(tensor_data2, tensor_data2.shape)



[[[0.5465580469336572, 0.04236814603546113, 0.5790113916490501, 0.30143791853472324, 0.9553697161260118, 0.7017564388672006, 0.12690651180140555, 0.2176322352004353, 0.7756344776809508, 0.18257625875585515, 0.6954844612327116, 0.4025771117991356, 0.5708745687703768, 0.9860208097508187, 0.7967220919932435, 0.5005413596767971, 0.5467156328379534, 0.03328665426059996, 0.33283155583253066, 0.32039811221194225, 0.2877663657324472, 0.207906608189383, 0.3407836819882911, 0.292875145257232, 0.08159483294180148, 0.4686540745530219, 0.5676066192031441, 0.0014052286668950753, 0.035527860136935874, 0.6969275144454442, 0.8845821586281839, 0.7813545590314249, 0.3520150843724008, 0.655580826143855, 0.8094879707265338, 0.5304716363245846, 0.1854333781967914, 0.13264826355665993, 0.7012679858496631, 0.04233387250264131, 0.6468801152361034, 0.42123193680160775, 0.22102943196571734, 0.18594211881826683, 0.8349594490172789, 0.471342346740133, 0.44254281011126106, 0.7990344939649782, 0.03403185101411843, 0

- torch.tensor() - always copy data 
- if numpy -> torch -> use torch.as_tensor(np_array)
- torch.detach() -> returns a new tensor that shares the same memory loc and wont copy
- torch the value, datatype and device can be mentioned in one go - torch.tensor([1], dtype=torch.int, device='cuda')

In [5]:
sample_tensor = torch.tensor([1], dtype=torch.int, device='cuda')
sample_tensor

tensor([1], device='cuda:0', dtype=torch.int32)

In [6]:
# just a single value, but a 3d tensor
x = torch.tensor([[[1]]])

# how to get the value?
x.item()


1

### Tensor basics

In [10]:
a = torch.zeros(2,3)
r, c = a.shape
assert a.dtype == torch.float32 # this is the default
assert r == 2
assert c == 3

In [ ]:
# using item

a2  = torch.zeros(3, 4, dtype=torch.float64) 
assert a2.shape == (3,4)
assert a2.dtype == torch.float64

assert a2[1,1].item() == 0

In [24]:
# creating a random array that has values betwenn (0,1]
# note: rand() is uniform distribution between [0, 1) 
# randn() has no distribution so it can be anything

rand1 = torch.rand(2,3)
assert torch.all((rand1 >= -1) & (rand1 <= 1))

rand2 = torch.randn(3,4)
assert torch.all((rand2 <= -1) | (rand2 > -1 )) # this will always hold true anyway

In [29]:
# arange vs range
# torch.arange runs on gpu,cpu, mps etc returns a tensor, while range is cpu only. 
# hence torch.arange is more flexible and we dont have to worry about the accelerator.

# regarding the range it works similar to range in python where the last is not included.
arr1 = torch.arange(0, 10, step=2) 
assert torch.equal(arr1, torch.Tensor([0,2,4,6,8]))

# other ways to do the above
assert arr1.tolist() == [0,2,4,6,8]

assert list(arr1) == [0,2,4,6,8]

In [ ]:
# linspace
# creates a one-dim space of size strps between the range provided.
arr3 = torch.linspace(0, 1, steps=10) # this will return a tensor of size 10, with values evenly spaced between 0 and 1
print(arr3)

# lets assert that the spaces are consistent
# getting the diffs between each value
arr3_diff = arr3.diff()
print(arr3_diff)

print(arr3_diff[0])
# we can next use a single assert that will broadcast and check this in a vectorized way
assert torch.allclose(arr3_diff.min(), arr3_diff.max())

# since this is a one dim array - we can use flip for 0 copy - here 0 is the dimension
# flip doesnt allocate new memory and it reverses the stride, giving zero copy view
# hence flip returns a view
# this has to be true - since the diff array is identical
assert torch.allclose(arr3_diff, arr3_diff.flip(0))

tensor([0.0000, 0.1111, 0.2222, 0.3333, 0.4444, 0.5556, 0.6667, 0.7778, 0.8889,
        1.0000])
tensor([0.1111, 0.1111, 0.1111, 0.1111, 0.1111, 0.1111, 0.1111, 0.1111, 0.1111])
tensor(0.1111)


## Reshaping

In [ ]:
t1 = torch.Tensor([[1,2,3], [4,5,6]])

print(t1) # original

t2 = t1.reshape((3,2))

print(t2) # may or may not copy, if the memory is not contigious

t3 = t1.view(3,2)

print(t3) # no copy

# reshape creates a new tensor if not contiguous
if t1.is_contiguous():
    assert t1.data_ptr() == t2.data_ptr()
else:
    assert t1.data_ptr() != t2.data_ptr()

# validating they are pointing to the same memory
assert t1.data_ptr() == t3.data_ptr() # since they point to same memory (surprising)

tensor([[1., 2., 3.],
        [4., 5., 6.]])
tensor([[1., 2.],
        [3., 4.],
        [5., 6.]])
tensor([[1., 2.],
        [3., 4.],
        [5., 6.]])


In [54]:
# here is an example - where we use transpose to make the tensor non-contegious
# then we reshape it to original dims and we can verify they are different arrays
t = torch.randn(3, 4)
t2 = t.T  # non-contiguous (transposed view)
assert t2.is_contiguous() == False
t3 = t2.reshape(3, 4)

assert t.data_ptr() != t3.data_ptr()

### Squeeze and Unsqueeze

In [10]:
arr = torch.rand(3,4)
print(arr.shape)

arr_usq = arr.unsqueeze(0) # adding another dim to the 0th axes, shape should be 1 x 3 x 4
print(arr_usq.shape)

arr_usq1 = arr_usq.unsqueeze(0) # adding another dim to the 0th axes, shape should be 1 x 1 x 3 x 4
print(arr_usq1.shape)

arr_sq = arr_usq1.squeeze(0)
print(arr_sq.shape)

torch.Size([3, 4])
torch.Size([1, 3, 4])
torch.Size([1, 1, 3, 4])
torch.Size([1, 3, 4])


### Broadcasting

In [ ]:
# Broadcast aligns from the right. 
# eg: (3,4) can be aligned with (), (4,) or (3,4) or (1,3,4)
a = torch.ones(1,3,4)
b = torch.ones(4)

print(a, a.shape)
print(b, b.shape)

c = a + b

print(c, c.shape)

b = torch.ones(3)
c1 = a + b [:, None] # hack ,  # this will add value from b to each row of a
print(c, c1.shape)

# 2. Broadcast in-place ops are picky
# eg: -=, +=, - will not work 
# use a.add_(b)

# Like methods
# ones_like
# zeros_like
# full_like(<tensor_name>, <value)
# rand_like - uniform distribution
# empty_link - same shape, un initialized
# randn_like - normal distribution

d = a.add_(torch.ones_like(a)) # a need to be same shape

tensor([[[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]]]) torch.Size([1, 3, 4])
tensor([1., 1., 1., 1.]) torch.Size([4])
tensor([[[2., 2., 2., 2.],
         [2., 2., 2., 2.],
         [2., 2., 2., 2.]]]) torch.Size([1, 3, 4])
tensor([[[2., 2., 2., 2.],
         [2., 2., 2., 2.],
         [2., 2., 2., 2.]]]) torch.Size([1, 3, 4])


In [27]:
print(type(torch.tensor([1,2,3])), type(torch.Tensor([1,2,3])))

<class 'torch.Tensor'> <class 'torch.Tensor'>


In [31]:
# Indexing (like NumPy)
x = torch.arange(12).reshape(3, 4)
print(x)
print(x[0])         # first row
print(x[:, 1])      # second column
print(x[0, 1:3])    # elements [1,2] of row 0
print(x[x > 5])     # boolean mask

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
tensor([0, 1, 2, 3])
tensor([1, 5, 9])
tensor([1, 2])
tensor([ 6,  7,  8,  9, 10, 11])
